# ARTEMIS — RAG Pipeline (local)

Corre en **local (Mac)**. No requiere GPU.

Produce:
- `retrieval_index.json` — entregable obligatorio
- `faiss.index` — índice para reutilizar en Colab
- `train_processed.pkl` / `val_processed.pkl` — datos con contexto pre-recuperado, listos para subir a Colab y entrenar

**Después de correr este notebook**, sube los `.pkl` y el `faiss.index` a Drive y abre `lora.ipynb` en Colab.

## 0 · Setup

In [34]:
# pip install -r requirements.txt  (solo la primera vez)

# Fijar env vars ANTES de importar torch — evita segfault en Apple Silicon (Mac)
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_JIT"]            = "0"   # deshabilita JIT (causa segfault en Mac)
os.environ["OMP_NUM_THREADS"]        = "1"   # single thread, más estable en Mac

import json
import re
import sys
import unicodedata
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
torch.backends.mps.is_available = lambda: False   # forzar CPU puro
from sklearn.model_selection import train_test_split

BASE   = Path(".")
DATA   = BASE / "Data"
KB_DIR = DATA / "knowledge_base" / "knowledge_base"

RANDOM_STATE = 42
RETRIEVAL_K  = 8    # cuántos candidatos recupera FAISS/BM25 internamente
PROMPT_K     = 5    # cuántos chunks se guardan en los PKLs (se usan en el prompt)

sys.path.insert(0, str(BASE))

print("Setup OK")

Setup OK


## 1 · Data Loading & Exploration

In [35]:
train_raw = pd.read_csv(DATA / "train.csv")
test_df   = pd.read_csv(DATA / "test.csv")

print(f"Train: {len(train_raw)} | Test: {len(test_df)}")
train_raw.head(3)

Train: 2718 | Test: 766


,id,query,tool_call
0,Q-00001,Cóndor radiation sensors are showing 2.1 mSv/h...,"activate_protocol(protocol_id='MASA-SEC-012',s..."
1,Q-00002,"We’ve got a rapid temperature rise in Quetzal,...","activate_protocol(protocol_id='MASA-SEC-003',s..."
2,Q-00003,Can we get the trajectory calculations for the...,"calculate_trajectory(maneuver='reentry',urgenc..."


In [36]:
with open(DATA / "tools_definition.json") as f:
    tools_def_raw = json.load(f)

TOOLS_DEF   = {t["name"]: t for t in tools_def_raw["tools"]}
VALID_TOOLS = list(TOOLS_DEF.keys())

print("Herramientas:")
for name, info in TOOLS_DEF.items():
    params = list(info.get("parameters", {}).keys())
    print(f"  {name}({', '.join(params)})")

Herramientas:
  get_telemetry(module, metric, timeframe_hours)
  get_crew_status(module, info)
  get_module_status(module, system)
  send_alert(module, severity, reason)
  send_message(recipient, priority)
  schedule_maintenance(module, task, priority)
  activate_protocol(protocol_id, scope)
  control_system(module, system, action)
  calculate_trajectory(maneuver, urgency)
  request_supply(category, urgency)
  no_action()


In [37]:
def extract_tool_name(tc: str) -> str:
    return tc.split("(")[0].strip()

train_raw["tool_name"] = train_raw["tool_call"].apply(extract_tool_name)
print("Distribución de tools:")
print(train_raw["tool_name"].value_counts().to_string())

Distribución de tools:
tool_name
activate_protocol       797
send_alert              675
no_action               331
control_system          162
calculate_trajectory    121
request_supply          121
schedule_maintenance    117
get_telemetry           107
get_module_status       104
get_crew_status          93
send_message             90


In [38]:
with open(DATA / "consultas_centro_control.json") as f:
    consultas = json.load(f)

kb_docs = sorted(KB_DIR.rglob("doc.md"))
print(f"Documentos KB: {len(kb_docs)} | Consultas para métricas: {len(consultas)}")

Documentos KB: 54 | Consultas para métricas: 810


## 2 · Preprocessing

In [39]:
df = train_raw.copy()

# Duplicados exactos
before = len(df)
df = df.drop_duplicates(subset=["query"])
print(f"Duplicados eliminados: {before - len(df)}")

# Tools inválidas
before = len(df)
df = df[df["tool_name"].isin(VALID_TOOLS)].copy()
print(f"Filas con tool inválida: {before - len(df)}")

# Normalizar queries
def normalize_query(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", text).strip()

df["query"] = df["query"].apply(normalize_query)
test_df["query"] = test_df["query"].apply(normalize_query)

print(f"\nDataset limpio: {len(df)} ejemplos")

Duplicados eliminados: 148
Filas con tool inválida: 0

Dataset limpio: 2570 ejemplos


In [40]:
train_df, val_df = train_test_split(
    df, test_size=0.10, random_state=RANDOM_STATE, stratify=df["tool_name"]
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Validation: {len(val_df)}")

Train: 2313 | Validation: 257


## 3 · Chunking + Embedding + FAISS Index

In [41]:
from scripts.chunker import SentenceTextSplitter, load_markdown_as_pages

splitter = SentenceTextSplitter(section_length=800, overlap_pct=0.20, max_tokens=400)

chunks = []
for doc_path in sorted(KB_DIR.rglob("doc.md")):
    doc_id = doc_path.parent.name
    pages  = load_markdown_as_pages(str(doc_path))
    for i, sp in enumerate(splitter.split_pages(pages)):
        chunks.append({"doc_id": doc_id, "chunk_id": i, "text": sp.text})

print(f"Total chunks: {len(chunks)}")

Total chunks: 785


In [42]:
from transformers import AutoTokenizer, AutoModel

# sentence_transformers causa segfault en Mac con este entorno de torch.
# Usamos transformers directo — mismo resultado, sin el crash.
_enc_tok = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")
_enc_mdl = AutoModel.from_pretrained("BAAI/bge-small-en-v1.5")
_enc_mdl.eval()

def encode(texts, batch_size=32):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = _enc_tok(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
        with torch.no_grad():
            out = _enc_mdl(**enc)
        vecs = torch.nn.functional.normalize(out.last_hidden_state[:, 0, :], p=2, dim=1)
        all_vecs.append(vecs.cpu().numpy())
    return np.concatenate(all_vecs, axis=0)

texts = [c["text"] for c in chunks]
vecs  = encode(texts)
print(f"Embeddings shape: {vecs.shape}")   # (785, 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (785, 384)


In [43]:
dim   = vecs.shape[1]  # 384
index = faiss.IndexFlatIP(dim)
index.add(vecs.astype(np.float32))

faiss.write_index(index, str(BASE / "faiss.index"))
print(f"FAISS: {index.ntotal} vectores guardados en faiss.index")

FAISS: 785 vectores guardados en faiss.index


In [44]:
# Guardar retrieval_index.json (entregable obligatorio)
retrieval_index = [
    {"doc_id": c["doc_id"], "chunk_id": c["chunk_id"],
     "text": c["text"], "vector": vecs[i].tolist()}
    for i, c in enumerate(chunks)
]
with open(BASE / "retrieval_index.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_index, f, ensure_ascii=False)

print("retrieval_index.json guardado.")

retrieval_index.json guardado.


In [45]:
# !pip install -q rank_bm25   # solo la primera vez (ya en requirements.txt)
from rank_bm25 import BM25Okapi

def _strip_accents(text: str) -> str:
    """'Cóndor' → 'condor' — normaliza para que BM25 matchee sin importar tildes."""
    nfd = unicodedata.normalize("NFD", text)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

def _bm25_tok(text: str) -> list:
    return re.findall(r'\w+', _strip_accents(text).lower())

bm25_index = BM25Okapi([_bm25_tok(c["text"]) for c in chunks])
print(f"BM25 index: {len(chunks)} documentos")


def retrieve(query: str, k: int = RETRIEVAL_K) -> list:
    """Dense-only retrieval con FAISS (para comparar con hybrid)."""
    enc = _enc_tok([query], padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        out = _enc_mdl(**enc)
    q_vec = torch.nn.functional.normalize(out.last_hidden_state[:, 0, :], p=2, dim=1).cpu().numpy().astype(np.float32)
    _, idxs = index.search(q_vec, k)
    return [chunks[i] for i in idxs[0] if i < len(chunks)]


def retrieve_hybrid(query: str, k: int = RETRIEVAL_K, rrf_k: int = 60) -> list:
    """Reciprocal Rank Fusion: FAISS dense + BM25 → top-k chunks.
    
    BM25 matchea términos exactos (IDs de protocolo, nombres de módulo).
    Dense matchea semántica / paráfrasis.
    RRF combina ambas listas sin necesitar calibración.
    """
    n      = len(chunks)
    k_cand = min(k * 4, n)

    # Dense FAISS
    enc = _enc_tok([query], padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        out = _enc_mdl(**enc)
    q_vec = torch.nn.functional.normalize(out.last_hidden_state[:, 0, :], p=2, dim=1).cpu().numpy().astype(np.float32)
    _, dense_idxs = index.search(q_vec, k_cand)
    dense_idxs = [i for i in dense_idxs[0] if i < n]

    # BM25 (con normalización de acentos)
    bm25_scores = bm25_index.get_scores(_bm25_tok(query))
    bm25_idxs   = np.argsort(bm25_scores)[::-1][:k_cand].tolist()

    # RRF: score = Σ 1/(rrf_k + rank)
    rrf = {}
    for rank, idx in enumerate(dense_idxs):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (rrf_k + rank + 1)
    for rank, idx in enumerate(bm25_idxs):
        rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (rrf_k + rank + 1)

    top = sorted(rrf, key=rrf.__getitem__, reverse=True)[:k]
    return [chunks[i] for i in top]


# Smoke test
sample = retrieve_hybrid("Condor radiation emergency protocol")
print(f"Hybrid top-1: {sample[0]['doc_id']} | {sample[0]['text'][:100]}...")

BM25 index: 785 documentos
Hybrid top-1: MASA-DOC-009 | 

Upon activation, the protocol enforces five mandatory actions in sequence: first, an automated sta...


## 4 · Métricas de Retrieval (P@K, R@K)

In [46]:
def retrieval_metrics(retrieve_fn, k: int = 3) -> float:
    hits = sum(
        1 for item in consultas
        if item["doc_id"] in [c["doc_id"] for c in retrieve_fn(normalize_query(item["query"]), k=k)]
    )
    return round(hits / len(consultas), 4)

print("Métricas de retrieval (P@K = R@K, 1 doc relevante por query)")
print(f"{'k':>3}  {'Dense':>8}  {'Hybrid':>8}  {'Δ':>7}")
print("-" * 34)
for k in [1, 3, 5, 8]:
    d = retrieval_metrics(retrieve,        k=k)
    h = retrieval_metrics(retrieve_hybrid, k=k)
    print(f"{k:>3}  {d:>8.4f}  {h:>8.4f}  {h - d:>+7.4f}")

Métricas de retrieval (P@K = R@K, 1 doc relevante por query)
  k     Dense    Hybrid        Δ
----------------------------------
  1    0.5704    0.6568  +0.0864
  3    0.7481    0.8358  +0.0877
  5    0.8148    0.9037  +0.0889
  8    0.8642    0.9358  +0.0716


## 5 · Pre-recuperar contexto para train/val → exportar para Colab

In [47]:
print("Generando contexto para train...")
train_df["context_chunks"] = train_df["query"].apply(lambda q: retrieve_hybrid(q, k=PROMPT_K))

print("Generando contexto para validation...")
val_df["context_chunks"] = val_df["query"].apply(lambda q: retrieve_hybrid(q, k=PROMPT_K))

print("Generando contexto para test...")
test_df["context_chunks"] = test_df["query"].apply(lambda q: retrieve_hybrid(q, k=PROMPT_K))

print("Listo.")

Generando contexto para train...
Generando contexto para validation...
Generando contexto para test...
Listo.


In [48]:
# Exportar DataFrames con contexto — subir estos a Colab/Drive
train_df.to_pickle(BASE / "train_processed.pkl")
val_df.to_pickle(BASE / "val_processed.pkl")
test_df.to_pickle(BASE / "test_processed.pkl")

print("Exportado:")
print(f"  train_processed.pkl — {len(train_df)} filas")
print(f"  val_processed.pkl   — {len(val_df)} filas")
print(f"  test_processed.pkl  — {len(test_df)} filas")
print()
print("Sube a Colab/Drive: train_processed.pkl, val_processed.pkl, test_processed.pkl")
print("Luego abre lora.ipynb en Colab.")

Exportado:
  train_processed.pkl — 2313 filas
  val_processed.pkl   — 257 filas
  test_processed.pkl  — 766 filas

Sube a Colab/Drive: train_processed.pkl, val_processed.pkl, test_processed.pkl
Luego abre lora.ipynb en Colab.
